# Notebook 03 — Credit Risk Model
## End-to-End Classification with Convergence Analysis

**Purpose:** Train the final model, evaluate classification performance,
and tie optimisation results back to the theoretical analysis.


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_preparation import load_raw_data, prepare_data
from src.gradient_descent import batch_gradient_descent
from src.evaluation import classification_metrics, optimisation_summary, compare_with_sklearn
from src.visualization import (
    plot_loss_curve, plot_grad_norm, plot_confusion_matrix,
    plot_eigenvalue_spectrum,
)

SEED = 42
np.random.seed(SEED)
%matplotlib inline


## 1. Data Preparation

In [ ]:
df = load_raw_data("../data/raw")
data = prepare_data(df, test_size=0.20, random_state=SEED, scale=True)

X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]

print(f"Features: {X_train.shape[1]}")
print(f"Train: {X_train.shape[0]}  (bad-credit rate: {y_train.mean():.1%})")
print(f"Test:  {X_test.shape[0]}   (bad-credit rate: {y_test.mean():.1%})")


## 2. Train Final Model

Using the best settings identified in Notebook 02:
- Feature scaling: StandardScaler
- Learning rate: η = 0.1  (adjust based on Experiment A results)
- Regularisation: λ = 0   (or adjust based on Experiment C)
- Stopping tolerance: 1e-6


In [ ]:
# Final training run
result = batch_gradient_descent(
    X_train, y_train,
    learning_rate=0.1,
    lam=0.0,
    max_iter=10000,
    tol=1e-6,
    verbose=True,
    print_every=500,
    compute_hessian=True,
)

opt_summary = optimisation_summary(result)
print("\n=== Optimisation Summary ===")
for k, v in opt_summary.items():
    print(f"  {k}: {v}")


## 3. Convergence Diagnostics

In [ ]:
fig = plot_loss_curve(result.loss_history,
                     title="Final Model: Loss vs Iteration",
                     save_name="final_loss_curve")
plt.show()


In [ ]:
fig = plot_grad_norm(result.grad_norm_history,
                    title="Final Model: Gradient Norm vs Iteration",
                    save_name="final_grad_norm")
plt.show()


In [ ]:
# Hessian diagnostics
print(f"Condition number κ(H): {result.condition_number:.2f}")

if result.hessian_eigenvalues is not None:
    print(f"λ_min = {result.hessian_eigenvalues[0]:.6f}")
    print(f"λ_max = {result.hessian_eigenvalues[-1]:.6f}")
    print(f"Lipschitz constant L ≈ λ_max = {result.hessian_eigenvalues[-1]:.6f}")
    print(f"Theoretical max step size ≈ 2/L = {2.0/result.hessian_eigenvalues[-1]:.4f}")

    fig = plot_eigenvalue_spectrum(result.hessian_eigenvalues,
                                  title="Final Model: Hessian Eigenvalue Spectrum",
                                  save_name="final_eigenvalues")
    plt.show()


## 4. Classification Performance

**Positive class:** y = 1 (bad credit)

> Note: Because the dataset has 30% bad-credit cases by design,
> accuracy alone is not sufficient — we report precision, recall, F1
> and ROC-AUC.


In [ ]:
# Training set metrics
train_metrics = classification_metrics(X_train, y_train, result.w, result.b)
print("=== TRAINING SET ===")
print(train_metrics["report"])
print(f"ROC-AUC: {train_metrics['roc_auc']:.4f}")


In [ ]:
# Test set metrics
test_metrics = classification_metrics(X_test, y_test, result.w, result.b)
print("=== TEST SET ===")
print(test_metrics["report"])
print(f"ROC-AUC: {test_metrics['roc_auc']:.4f}")


In [ ]:
# Confusion matrix
fig = plot_confusion_matrix(
    test_metrics["confusion_matrix"],
    title="Test Set Confusion Matrix",
    save_name="final_confusion_matrix",
)
plt.show()


## 5. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc
from src.logistic_regression import predict_proba

y_prob = predict_proba(X_test, result.w, result.b)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, linewidth=2, label=f"GD Model (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Test Set")
ax.legend(loc="lower right")
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig("../results/figures/roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. sklearn Baseline Comparison

In [ ]:
comparison = compare_with_sklearn(
    X_train, y_train, X_test, y_test,
    w_gd=result.w, b_gd=result.b, lam=0.0,
)

print(f"Our GD accuracy (test):     {test_metrics['accuracy']:.4f}")
print(f"sklearn accuracy (test):    {comparison['sklearn_accuracy']:.4f}")
print(f"Max coefficient difference: {comparison['max_w_diff']:.6f}")
print(f"Max probability difference: {comparison['max_prob_diff']:.6f}")


## 7. Results Summary

| Metric | Training | Test |
|--------|----------|------|
| Accuracy | {train_acc} | {test_acc} |
| Precision | {train_prec} | {test_prec} |
| Recall | {train_rec} | {test_rec} |
| F1-score | {train_f1} | {test_f1} |
| ROC-AUC | {train_auc} | {test_auc} |

*Fill in the values from the outputs above.*

### Key Observations

1. **Convergence:** The model converged in ___ iterations with final loss ___ and gradient norm ___.
2. **Conditioning:** κ(H) = ___, indicating ___.
3. **Classification:** The model achieves ___ ROC-AUC on the test set.
4. **Theory vs practice:** The observed convergence rate is consistent / inconsistent with the theoretical prediction because ___.

### Limitations

- The dataset is from 1973–1975 and does not represent current lending conditions.
- The 30% bad-credit rate is by design, not a natural base rate.
- This is a mathematical study; the model is not suitable for deployment.
